In [ ]:
# =========================
# IMPORTURI GLOBALE & CONFIG
# =========================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from packaging import version

# Scikit-learn
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.feature_selection import SelectKBest, mutual_info_regression, SelectFromModel
from sklearn.linear_model import LinearRegression, Ridge, Lasso, QuantileRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("Importuri OK — scikit-learn", version.parse(__import__("sklearn").__version__))

DATA_PATH = Path("train_split.csv")
EVAL_PATH = Path("eval_split.csv")
TIME_COL = "data_ora"
TARGET = "total"


In [ ]:
# =========================
# ÎNCĂRCARE DATE + FEAT ENGINEERING DE BAZĂ
# =========================
def load_bike_frame(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    df["ora"] = df[TIME_COL].dt.hour
    df["zi_saptamana"] = df[TIME_COL].dt.dayofweek
    df["luna"] = df[TIME_COL].dt.month
    df["an"] = df[TIME_COL].dt.year
    df["saptamana"] = df[TIME_COL].dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["zi_saptamana"] >= 5).astype(int)
    df["sin_ora"] = np.sin(2 * np.pi * df["ora"] / 24)
    df["cos_ora"] = np.cos(2 * np.pi * df["ora"] / 24)
    return df

df_raw = load_bike_frame(DATA_PATH)
df_raw.set_index(TIME_COL, inplace=True)
print("Dimensiuni:", df_raw.shape)
df_raw.head()


## PART 1 - Exploratory Data Analysis (EDA)



### Analiza 1 - Structura setului & valori lipsă

Înțelegerea tipurilor de date și a valorilor lipsă dictează pașii de preprocesare (imputare/encodare) și confirmă dacă putem folosi modele sensibile la `NaN`.

Statistici descriptive (`describe`), tipurile de coloane (`info`) și o sinteză numerică a valorilor lipsă.


In [ ]:
display(df_raw.head())
print("\n=== INFO ===")
df_raw.info()

missing_summary = df_raw.isna().sum()
missing_summary = missing_summary[missing_summary > 0]
print("\n=== COL. CU VALORI LIPSE ===")
print(missing_summary if not missing_summary.empty else "Nu există valori lipsă.")

df_raw.describe().T


### Analiza 2 - Distribuția țintei `total`

Modelele de regresie sunt sensibile la distribuții foarte asimetrice și outlieri. Identificarea formei distribuției ne spune dacă vom avea nevoie de metrici robuste (MAE, cuantile) și justifică folosirea modelelor cu cuantile.

Histogramă + boxplot pentru `total`, plus câteva cuantile relevante.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_raw[TARGET], bins=50, color="#4c78a8", edgecolor="white")
axes[0].set_title("Histogramă total")
axes[0].set_xlabel("biciclete/oră")
axes[0].set_ylabel("frecvență")
axes[1].boxplot(df_raw[TARGET], vert=True)
axes[1].set_title("Boxplot total")
plt.tight_layout()
plt.show()

quantiles = df_raw[TARGET].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).rename("quantile")
distrib_summary = pd.DataFrame({"value": quantiles})
distrib_summary


### Analiza 3 - Serie de timp & trend sezonier

Cerința impune o vizualizare ca serie de timp pentru a evidenția trenduri/ciclicități. Acest lucru justifică folosirea split-ului time-aware și a modelelor capabile să surprindă sezonalitatea.

Volumul zilnic agregat și media mobilă pe 7 zile; într-un al doilea plot surprind sezonalitatea orară și cea pe zilele săptămânii.


In [ ]:
daily = df_raw[TARGET].resample('D').sum()
rolling = daily.rolling(7).mean()
plt.figure(figsize=(14,4))
plt.plot(daily.index, daily, label='Total zilnic')
plt.plot(rolling.index, rolling, label='Media mobilă 7 zile', linewidth=2)
plt.title('Trend zilnic + media mobilă (7 zile)')
plt.xlabel('Data'); plt.ylabel('biciclete/zi'); plt.legend(); plt.show()

pivot_hour = df_raw.pivot_table(values=TARGET, index='ora', columns='zi_saptamana', aggfunc='mean')
plt.figure(figsize=(10,5))
sns.heatmap(pivot_hour, cmap='viridis')
plt.title('Intensitate medie pe oră vs zi a săptămânii')
plt.xlabel('zi_saptamana (0=Luni)'); plt.ylabel('ora');
plt.show()


### Analiza 4 - Corelații cu vremea și sezonalitatea

Relația dintre condițiile meteo și cerere este esențială pentru a decide ce atribute păstrăm/transformăm. O matrice de corelație plus boxplot-uri pe categorii (`sezon`, `vreme`) clarifică puterea predictivă.

Heatmap Pearson pentru numeric + boxplot `total` vs. `vreme`.


In [ ]:
num_cols = ["temperatura", "temperatura_resimtita", "umiditate", "viteza_vant", TARGET]
plt.figure(figsize=(8,6))
sns.heatmap(df_raw[num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Corelații Pearson între variabilele meteo și total")
plt.show()

plt.figure(figsize=(10,4))
sns.boxplot(data=df_raw, x="vreme", y=TARGET, palette="Set2")
plt.title("Distribuția total în funcție de condiția meteo (vreme)")
plt.show()


**Concluzii EDA:**
- Setul nu are valori lipsă, dar mixul de variabile numerice/categorice confirmă nevoia de pipelines separate.
- Distribuția țintei este puternic asimetrică => folosim MAE, cuantile și raportăm RMSE doar pentru comparabilitate.
- Seria temporală arată trend sezonier și patternuri săptămânale, justificând split-ul time-aware și adăugarea de feature-uri calendaristice/ciclice.
- Temperatură și umiditate au corelație moderată cu `total`, în timp ce condiția `vreme` și `sezon` explică variația prin schimbări ale medianei — utile pentru encodare + discretizare.


## PART 2 - Preprocesare (imputare, standardizare, encodare, discretizare, selecție a atributelor)

Pașii următori urmează recomandările enunțului:
1. **Curățarea setului** și definirea atributelor (eliminăm `ocazionali`, `inregistrati` pentru a evita scurgerea țintei).
2. **Split** time-aware 80/20 (primii 80% pentru train, restul pentru valid).
3. **Pipeline** care include imputare, standardizare, encodare OHE/ordinală și discretizarea temperaturii.
4. **Selecția atributelor** cu `SelectKBest` (mutual_info) și `SelectFromModel` (Lasso) pentru a păstra doar feature-urile utile.


In [ ]:
# =========================
# Curățare + split time-aware 80/20
# =========================
FEATURE_BLACKLIST = {TARGET, "ocazionali", "inregistrati"}
feature_cols = [c for c in df_raw.columns if c not in FEATURE_BLACKLIST]

df_model = df_raw[feature_cols + [TARGET]].copy()

n_total = len(df_model)
split_idx = int(np.floor(0.8 * n_total))
train_df = df_model.iloc[:split_idx]
valid_df = df_model.iloc[split_idx:]

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_valid = valid_df.drop(columns=[TARGET])
y_valid = valid_df[TARGET]

print(f"Train: {X_train.shape}, Valid: {X_valid.shape}")


In [ ]:
# =========================
# ColumnTransformer: imputare + standardizare + encodare + discretizare
# =========================
NUMERIC_COLS = [
    "temperatura", "temperatura_resimtita", "umiditate", "viteza_vant",
    "ora", "zi_saptamana", "luna", "an", "saptamana",
    "sin_ora", "cos_ora", "is_weekend"
]
CATEGORICAL_COLS = ["sezon", "sarbatoare", "zi_lucratoare"]
ORDINAL_COLS = ["vreme"] 
DISCRETIZE_COLS = ["temperatura"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("pass", "passthrough")
])

# Discretizare temperatură în 5 bin-uri (quantile) + OHE
kbins = KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="quantile")
discret_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("kbins", kbins)
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_COLS),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
        ("ord", ordinal_pipeline, ORDINAL_COLS),
        ("temp_bins", discret_pipeline, DISCRETIZE_COLS)
    ],
    remainder="drop",
    sparse_threshold=0.0
)

# Fit pentru a afla nr. de feature-uri
preprocess.fit(X_train)
preprocessed_sample = preprocess.transform(X_train[:5])
print("Pipeline preprocesare fit-uit (shape example):", preprocessed_sample.shape)


In [ ]:
X_train_proc = preprocess.transform(X_train)
X_valid_proc = preprocess.transform(X_valid)
print("Shapes după preprocess:", X_train_proc.shape, X_valid_proc.shape)


In [ ]:
# =========================
# Selecția atributelor (SelectKBest MI + SelectFromModel Lasso)
# =========================
sel_results = []
sel_transforms = {}

p = X_train_proc.shape[1]
ks = sorted({20, 30, 40, 60, 80, int(0.3*p), int(0.5*p)})
ks = [k for k in ks if 10 <= k <= p]

for k in ks:
    selector = SelectKBest(mutual_info_regression, k=k)
    Xtr_sel = selector.fit_transform(X_train_proc, y_train)
    Xva_sel = selector.transform(X_valid_proc)
    ridge = Ridge(alpha=0.1, random_state=RANDOM_STATE)
    ridge.fit(Xtr_sel, y_train)
    y_pred = ridge.predict(Xva_sel)
    mse = mean_squared_error(y_valid, y_pred)
    rmse = np.sqrt(mse)
    sel_results.append({"config": f"SelectKBest(k={k})", "n_features": k, "Valid_RMSE": rmse})
    sel_transforms[f"SelectKBest(k={k})"] = (selector, Xtr_sel, Xva_sel)

for alpha in [0.001, 0.01, 0.05, 0.1, 0.3, 0.7, 1.0]:
    lasso = Lasso(alpha=alpha, max_iter=20000, random_state=RANDOM_STATE)
    sfm = SelectFromModel(lasso, threshold="mean")
    Xtr_sel = sfm.fit_transform(X_train_proc, y_train)
    Xva_sel = sfm.transform(X_valid_proc)
    ridge = Ridge(alpha=0.1, random_state=RANDOM_STATE)
    ridge.fit(Xtr_sel, y_train)
    pred = ridge.predict(Xva_sel)
    mse = mean_squared_error(y_valid, pred)
    rmse = np.sqrt(mse)
    sel_results.append({"config": f"SelectFromModel Lasso alpha={alpha}", "n_features": Xtr_sel.shape[1], "Valid_RMSE": rmse})
    sel_transforms[f"SelectFromModel Lasso alpha={alpha}"] = (sfm, Xtr_sel, Xva_sel)

sel_df = pd.DataFrame(sel_results).sort_values("Valid_RMSE").reset_index(drop=True)
sel_df


In [ ]:
best_sel = sel_df.iloc[0]
best_name = best_sel["config"]
best_selector, X_train_sel, X_valid_sel = sel_transforms[best_name]
print(f"Selector ales: {best_name} | n_features={X_train_sel.shape[1]} | RMSE_valid={best_sel['Valid_RMSE']:.2f}")


## PART 3 · Modele și evaluare (cerința 4.3)

Toate modelele folosesc același pipeline (preprocess + selector ales) pentru a evita scurgerile. Hiper-parametrii sunt căutați prin TimeSeriesSplit (5 fold-uri) pe setul de train, iar performanța este raportată atât pe CV (media MSE), cât și pe setul de validare (MSE, MAE, RMSE, R²).


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

def evaluate_model(name, estimator, param_grid=None, n_iter=10):
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("selector", best_selector),
        ("model", estimator)
    ])
    if param_grid:
        search = RandomizedSearchCV(
            pipeline,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=tscv,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True,
            return_train_score=False
        )
        search.fit(X_train, y_train)
        best_estimator = search.best_estimator_
        cv_mse = -search.best_score_
        best_params = search.best_params_
    else:
        pipeline.fit(X_train, y_train)
        best_estimator = pipeline
        cv_scores = []
        for tr_idx, te_idx in tscv.split(X_train):
            X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
            y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]
            pipeline.fit(X_tr, y_tr)
            preds = pipeline.predict(X_te)
            cv_scores.append(mean_squared_error(y_te, preds))
        cv_mse = np.mean(cv_scores)
        best_params = {}

    y_pred = best_estimator.predict(X_valid)
    mse = mean_squared_error(y_valid, y_pred)
    mae = mean_absolute_error(y_valid, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_valid, y_pred)
    return {
        "model": name,
        "CV_MSE": cv_mse,
        "Valid_MSE": mse,
        "Valid_MAE": mae,
        "Valid_RMSE": rmse,
        "Valid_R2": r2,
        "best_estimator": best_estimator,
        "best_params": best_params
    }


In [ ]:
best_models = {}
results_rows = []


In [ ]:
# ===== Linear Regression =====
lin_configs = [
    ("LinearRegression", LinearRegression()),
    ("LinearRegression (no intercept)", LinearRegression(fit_intercept=False)),
    ("LinearRegression (positive)", LinearRegression(positive=True))
]
lin_rows = []
for name, estimator in lin_configs:
    res = evaluate_model(name, estimator)
    lin_rows.append(res)

lin_df = pd.DataFrame([{k:v for k,v in r.items() if k not in {"best_estimator","best_params"}} for r in lin_rows])
lin_df.sort_values("Valid_MSE")

best_lin = min(lin_rows, key=lambda r: r["Valid_MSE"])
best_models["LinearRegression"] = best_lin
results_rows.append({k:v for k,v in best_lin.items() if k not in {"best_estimator","best_params"}})
print("Best Linear Regression:", best_lin["model"], "| Valid_MSE=", round(best_lin["Valid_MSE"],2))


In [ ]:
# ===== SVR =====
svr_param_dist = {
    "model__kernel": ["linear", "rbf"],
    "model__C": [0.3, 1, 3, 10, 30, 100],
    "model__epsilon": [0.05, 0.1, 0.2, 0.3],
    "model__gamma": ["scale", 0.001, 0.01, 0.03, 0.1]
}
svr_result = evaluate_model("SVR", SVR(), param_grid=svr_param_dist, n_iter=25)
best_models["SVR"] = svr_result
results_rows.append({k:v for k,v in svr_result.items() if k not in {"best_estimator","best_params"}})
print("SVR best params:", svr_result["best_params"])
print("Valid_MSE=", round(svr_result["Valid_MSE"],2))


In [ ]:
# ===== RandomForestRegressor =====
rf_param_dist = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [None, 8, 12, 16, 24],
    "model__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 4, 6],
    "model__bootstrap": [True, False]
}
rf_result = evaluate_model("RandomForest", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), param_grid=rf_param_dist, n_iter=40)
best_models["RandomForest"] = rf_result
results_rows.append({k:v for k,v in rf_result.items() if k not in {"best_estimator","best_params"}})
print("RF best params:", rf_result["best_params"])
print("Valid_MSE=", round(rf_result["Valid_MSE"],2))


In [ ]:
# ===== GradientBoostingRegressor (loss = squared_error) =====
gb_param_dist = {
    "model__n_estimators": [200, 300, 400, 600],
    "model__learning_rate": [0.03, 0.05, 0.07, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": ["sqrt", "log2", 0.5, None]
}
gb_result = evaluate_model("GBR (squared)", GradientBoostingRegressor(loss="squared_error", random_state=RANDOM_STATE), param_grid=gb_param_dist, n_iter=30)
best_models["GBR_squared"] = gb_result
results_rows.append({k:v for k,v in gb_result.items() if k not in {"best_estimator","best_params"}})
print("GBR squared best params:", gb_result["best_params"])
print("Valid_MSE=", round(gb_result["Valid_MSE"],2))


In [ ]:
# ===== GradientBoostingRegressor (loss = quantile, α ∈ {0.10, 0.50, 0.90}) =====
quantile_results = {}
quantile_preds = {}
quantile_param_dist = {
    "model__n_estimators": [300, 400, 600],
    "model__learning_rate": [0.03, 0.05, 0.07, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": ["sqrt", "log2", 0.5, None]
}

for alpha in [0.10, 0.50, 0.90]:
    estimator = GradientBoostingRegressor(loss="quantile", alpha=alpha, random_state=RANDOM_STATE)
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("selector", best_selector),
        ("model", estimator)
    ])
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=quantile_param_dist,
        n_iter=20,
        cv=tscv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True
    )
    search.fit(X_train, y_train)
    best_est = search.best_estimator_
    preds = best_est.predict(X_valid)
    quantile_results[alpha] = {
        "best_estimator": best_est,
        "best_params": search.best_params_,
        "CV_MSE": -search.best_score_,
        "Valid_MSE": mean_squared_error(y_valid, preds),
        "Valid_MAE": mean_absolute_error(y_valid, preds),
        "Valid_R2": r2_score(y_valid, preds)
    }
    quantile_preds[alpha] = preds
    print(f"alpha={alpha} | best params: {search.best_params_}")

# metrici principale pe mediană + acoperirea benzii
median_metrics = quantile_results[0.50]
coverage = np.mean((y_valid.values >= quantile_preds[0.10]) & (y_valid.values <= quantile_preds[0.90]))

best_models["GBR_quantile"] = {
    "model": "GBR quantile (median)",
    "best_estimator": quantile_results[0.50]["best_estimator"],
    "best_params": quantile_results[0.50]["best_params"],
    "Valid_MSE": median_metrics["Valid_MSE"],
    "Valid_MAE": median_metrics["Valid_MAE"],
    "Valid_RMSE": np.sqrt(median_metrics["Valid_MSE"]),
    "Valid_R2": median_metrics["Valid_R2"],
    "CV_MSE": median_metrics["CV_MSE"],
    "coverage": coverage
}

results_rows.append({
    "model": "GBR quantile (median)",
    "CV_MSE": median_metrics["CV_MSE"],
    "Valid_MSE": median_metrics["Valid_MSE"],
    "Valid_MAE": median_metrics["Valid_MAE"],
    "Valid_RMSE": np.sqrt(median_metrics["Valid_MSE"]),
    "Valid_R2": median_metrics["Valid_R2"]
})
print(f"GBR quantile median — coverage [0.10,0.90] = {coverage:.1%}")


In [ ]:
# ===== QuantileRegressor (τ ∈ {0.10, 0.50, 0.90}) =====
qr_results = {}
alpha_grid = [0.0, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.3, 0.7, 1.0, 3.0]

for tau in [0.10, 0.50, 0.90]:
    estimator = QuantileRegressor(quantile=tau, solver="highs", fit_intercept=True)
    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("selector", best_selector),
        ("model", estimator)
    ])
    search = RandomizedSearchCV(
        pipeline,
        param_distributions={"model__alpha": alpha_grid},
        n_iter=min(8, len(alpha_grid)),
        cv=tscv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True
    )
    search.fit(X_train, y_train)
    best_est = search.best_estimator_
    preds = best_est.predict(X_valid)
    qr_results[tau] = {
        "best_estimator": best_est,
        "best_params": search.best_params_,
        "CV_MSE": -search.best_score_,
        "Valid_MSE": mean_squared_error(y_valid, preds),
        "Valid_MAE": mean_absolute_error(y_valid, preds),
        "Valid_R2": r2_score(y_valid, preds)
    }
    print(f"τ={tau} | best params: {search.best_params_}")

qr_median = qr_results[0.50]
coverage_qr = np.mean((y_valid.values >= qr_results[0.10]["best_estimator"].predict(X_valid)) &
                      (y_valid.values <= qr_results[0.90]["best_estimator"].predict(X_valid)))

best_models["QuantileRegressor"] = {
    "model": "QuantileRegressor (median)",
    "best_estimator": qr_median["best_estimator"],
    "best_params": qr_median["best_params"],
    "Valid_MSE": qr_median["Valid_MSE"],
    "Valid_MAE": qr_median["Valid_MAE"],
    "Valid_RMSE": np.sqrt(qr_median["Valid_MSE"]),
    "Valid_R2": qr_median["Valid_R2"],
    "CV_MSE": qr_median["CV_MSE"],
    "coverage": coverage_qr
}

results_rows.append({
    "model": "QuantileRegressor (median)",
    "CV_MSE": qr_median["CV_MSE"],
    "Valid_MSE": qr_median["Valid_MSE"],
    "Valid_MAE": qr_median["Valid_MAE"],
    "Valid_RMSE": np.sqrt(qr_median["Valid_MSE"]),
    "Valid_R2": qr_median["Valid_R2"]
})
print(f"QuantileRegressor median — coverage [0.10,0.90] = {coverage_qr:.1%}")


In [ ]:
results_df = pd.DataFrame(results_rows).set_index("model").sort_values("Valid_MSE")
results_df


### Vizualizări suplimentare: predicții vs. realitate

Mai jos:
1. Suprapun valorile reale cu predicțiile celor mai bune 3 modele de pe setul de validare.
2. Afișez un scatter `y_real` vs. `y_pred` pentru cel mai bun model (ușor de interpretat ca deviații față de diagonala ideală).
3. Prezint banda de predicție [0.10, 0.90] generată de GradientBoostingRegressor `loss="quantile"`, pentru a evidenția intervalul de incertitudine.



In [ ]:
def _get_estimator_by_label(label: str):
    for info in best_models.values():
        if info.get("model") == label:
            return info["best_estimator"]
    raise KeyError(f"Nu găsesc estimatorul pentru {label}.")

viz_labels = list(results_df.index[:3])
fig, axes = plt.subplots(len(viz_labels), 1, figsize=(12, 3.2 * len(viz_labels)), sharex=True)
if len(viz_labels) == 1:
    axes = [axes]

for ax, label in zip(axes, viz_labels):
    est = clone(_get_estimator_by_label(label))
    est.fit(X_train, y_train)
    preds = est.predict(X_valid)
    ax.plot(y_valid.index, y_valid, label="Real", color="#333")
    ax.plot(y_valid.index, preds, label=f"Pred {label}")
    ax.set_title(f"Validare: {label}")
    ax.set_ylabel("biciclete/oră")
    ax.legend(loc="upper left")

plt.xlabel("Timp (set valid)")
plt.tight_layout()
plt.show()

best_label = results_df.index[0]
best_est = clone(_get_estimator_by_label(best_label))
best_est.fit(X_train, y_train)
best_preds = best_est.predict(X_valid)

plt.figure(figsize=(6,6))
plt.scatter(y_valid, best_preds, alpha=0.4)
lims = [min(y_valid.min(), best_preds.min()), max(y_valid.max(), best_preds.max())]
plt.plot(lims, lims, color='red', linestyle='--')
plt.xlabel('Real total')
plt.ylabel(f'Pred total ({best_label})')
plt.title('Scatter real vs. pred — cel mai bun model (valid)')
plt.grid(True)
plt.show()

quantile_band = {}
for alpha in [0.10, 0.50, 0.90]:
    est = clone(quantile_results[alpha]["best_estimator"])
    est.fit(X_train, y_train)
    quantile_band[alpha] = est.predict(X_valid)

plt.figure(figsize=(12,4))
plt.plot(y_valid.index, y_valid, label='Real', color='#111')
plt.plot(y_valid.index, quantile_band[0.50], label='Mediană (α=0.50)', color='#1f77b4')
plt.fill_between(
    y_valid.index,
    quantile_band[0.10],
    quantile_band[0.90],
    color='#1f77b4', alpha=0.25, label='Bandă [0.10, 0.90]'
)
plt.title('GBR Quantile — bandă de predicție pe validare')
plt.xlabel('Timp (set valid)'); plt.ylabel('biciclete/oră'); plt.legend(loc='upper left')
plt.tight_layout(); plt.show()


## PART 4 - Evaluare finală pe `eval_split.csv`

După alegerea hiper-parametrilor, re-antrenez fiecare model pe combinația (train + valid) și raportez metricile pe `eval_split.csv` (care conține target). Pentru modelele pe cuantile raportez și acoperirea benzii [0.10, 0.90].


In [ ]:
df_eval = load_bike_frame(EVAL_PATH)
df_eval.set_index(TIME_COL, inplace=True)
df_eval_model = df_eval[feature_cols + [TARGET]].copy()
X_eval = df_eval_model.drop(columns=[TARGET])
y_eval = df_eval_model[TARGET]

X_full = pd.concat([X_train, X_valid])
y_full = pd.concat([y_train, y_valid])
print("Eval shape:", X_eval.shape)
print("Interval eval:", X_eval.index.min(), "→", X_eval.index.max())


In [ ]:
eval_rows = []

def compute_metrics(label, preds):
    mse = mean_squared_error(y_eval, preds)
    mae = mean_absolute_error(y_eval, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_eval, preds)
    eval_rows.append({
        "model": label,
        "Eval_MSE": mse,
        "Eval_MAE": mae,
        "Eval_RMSE": rmse,
        "Eval_R2": r2
    })

# Modele clasice
for key in ["LinearRegression", "SVR", "RandomForest", "GBR_squared"]:
    info = best_models[key]
    estimator = info["best_estimator"]
    estimator.fit(X_full, y_full)
    preds = estimator.predict(X_eval)
    compute_metrics(info["model"] if "model" in info else key, preds)

# GBR quantile (median + bandă)
for alpha in [0.10, 0.50, 0.90]:
    quantile_results[alpha]["best_estimator"].fit(X_full, y_full)
    quantile_preds[alpha] = quantile_results[alpha]["best_estimator"].predict(X_eval)

coverage_eval_gb = np.mean((y_eval.values >= quantile_preds[0.10]) & (y_eval.values <= quantile_preds[0.90]))
compute_metrics("GBR quantile (median)", quantile_preds[0.50])

# QuantileRegressor
for tau in [0.10, 0.50, 0.90]:
    qr_results[tau]["best_estimator"].fit(X_full, y_full)
    qr_results[tau]["preds_eval"] = qr_results[tau]["best_estimator"].predict(X_eval)

coverage_eval_qr = np.mean((y_eval.values >= qr_results[0.10]["preds_eval"]) & (y_eval.values <= qr_results[0.90]["preds_eval"]))
compute_metrics("QuantileRegressor (median)", qr_results[0.50]["preds_eval"])

print(f"GBR quantile coverage eval: {coverage_eval_gb:.1%}")
print(f"QuantileRegressor coverage eval: {coverage_eval_qr:.1%}")

eval_df = pd.DataFrame(eval_rows).set_index("model").sort_values("Eval_MSE")
eval_df


In [ ]:
best_eval_name = eval_df.index[0]
print("Cel mai bun model pe eval:", best_eval_name)

# reconstruim estimatorul corespunzător
if best_eval_name == "LinearRegression":
    estimator = best_models["LinearRegression"]["best_estimator"]
elif best_eval_name.startswith("SVR"):
    estimator = best_models["SVR"]["best_estimator"]
elif best_eval_name.startswith("RandomForest"):
    estimator = best_models["RandomForest"]["best_estimator"]
elif "GBR (squared" in best_eval_name:
    estimator = best_models["GBR_squared"]["best_estimator"]
elif "GBR quantile" in best_eval_name:
    estimator = quantile_results[0.50]["best_estimator"]
elif "QuantileRegressor" in best_eval_name:
    estimator = qr_results[0.50]["best_estimator"]
else:
    estimator = best_models["RandomForest"]["best_estimator"]

estimator.fit(X_full, y_full)
pred_eval = estimator.predict(X_eval)
pred_df = pd.DataFrame({
    "data_ora": X_eval.index,
    "pred_total": pred_eval,
    "real_total": y_eval.values
})
pred_path = Path("pred_eval_best_model.csv")
pred_df.to_csv(pred_path, index=False)
print("Predicții salvate în", pred_path.resolve())


## Concluzii

- **EDA** a identificat sezonalitate clară și o distribuție asimetrică a țintei => necesar să raportăm atât RMSE/MAE, cât și cuantile.
- **Preprocesarea** include toți pașii ceruți: imputare (mediană / most_frequent), standardizare pentru numerice, OneHot + ordinal pentru categorice, discretizarea temperaturii (KBins) și selecția atributelor (SelectKBest). Toți pașii sunt integrați într-un `Pipeline` reutilizabil.
- **Modele**: pentru fiecare algoritm a fost derulat TimeSeriesSplit + RandomizedSearch
- **Performanță**: conform `eval_df`, modelul cu cel mai mic MSE a fost folosit pentru generarea fișierului `pred_eval_best_model.csv`; pentru cuantile raportăm și acoperirea benzii.


